# Simulación de Dinámica Molecular de Complejos Proteína-Ligando con GROMACS en Google Colab

Este cuaderno de Jupyter interactivo ha sido diseñado para ejecutar **Simulaciones de Dinámica Molecular (MD) de 100 ns** utilizando **GROMACS** y **ACPYPE** en Google Colab. Este flujo de trabajo está optimizado para estudiar la estabilidad estructural y termodinámica de cuatro complejos proteína-ligando clave involucrados en el proyecto de investigación sobre resistencia a la meticilina en *Staphylococcus aureus* (MRSA):

1. **complex_MurG_Afzelin.pdb** (Modelo AlphaFold Q6GGZ0 + Pose de acoplamiento de Afzelina)
2. **complex_MurG_Quercetin.pdb** (Modelo AlphaFold Q6GGZ0 + Pose de acoplamiento de Quercetina - Control positivo)
3. **complex_PBP2a_Afzelin.pdb** (Receptor cristalino PBP2a 3ZG0_clean + Pose de acoplamiento de Afzelina)
4. **complex_PBP2a_Ceftaroline.pdb** (Receptor cristalino PBP2a 3ZG0_clean + Pose de acoplamiento de Ceftarolina - Control positivo)

---
### Flujo de Trabajo Metodológico
```
1. Configuración de Entorno (Mamba + GROMACS + ACPYPE + AmberTools)
                            ↓
2. Topología del Ligando (Campos de fuerza GAFF2 + Cargas AM1-BCC con ACPYPE)
                            ↓
3. Topología de la Proteína (Campo de fuerza AMBER99SB-ILDN con GROMACS pdb2gmx)
                            ↓
4. Ensamblaje del Complejo (Unión de archivos de coordenadas .gro y edición de topol.top)
                            ↓
5. Solvatación y Neutralización (Caja Dodecaédrica + Agua TIP3P + 0.15 M de NaCl)
                            ↓
6. Simulaciones MD (Minimización Energética -> Equilibrio NVT -> Equilibrio NPT -> Producción de 100 ns)
                            ↓
7. Post-procesamiento y Análisis (RMSD, RMSF, Radio de Giro, Enlaces de Hidrógeno + Visualización)
```
---
> [!IMPORTANT]
> Se recomienda encarecidamente utilizar un entorno de ejecución con **GPU** en Google Colab (vaya a *Entorno de ejecución -> Cambiar tipo de entorno de ejecución* y seleccione **T4 GPU** u otra disponible). Esto reducirá significativamente los tiempos de simulación.


## Paso 1: Configuración del Entorno en Google Colab

En este paso inicial, instalaremos **Conda/Mamba** a través de `condacolab` para administrar de forma limpia las dependencias científicas requeridas: **GROMACS**, **ACPYPE**, **AmberTools** (para antechamber/sqm) y **OpenBabel** (para conversión de archivos).


In [1]:
# 1. Instalar CondaColab en el entorno virtual de Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()


⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:13
🔁 Restarting kernel...


In [7]:
!rm -f /usr/local/conda-meta/pinned

In [8]:
# 2. Verificar la instalación de Conda e instalar herramientas científicas
import condacolab
condacolab.check()

# Instalar GROMACS, ACPYPE, OpenBabel y AmberTools desde conda-forge
!mamba install -y -c conda-forge gromacs acpype openbabel ambertools matplotlib pandas numpy


✨🍰✨ Everything looks OK!

Looking for: ['gromacs', 'acpype', 'openbabel', 'ambertools', 'matplotlib', 'pandas', 'numpy']

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache

Pinned packages:
  - python 3.11.*


Transaction

  Prefix: /usr/local

  Updating specs:

   - gromacs
   - acpype
   - openbabel
   - ambertools
   - matplotlib
   - pandas
   - numpy
   - ca-certificates
   - certifi
   - openssl


  Package                          Version  Build                              Channel           Size
───────────────────────────────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────────────────────────────

  + icu                               75.1  he02047a_0                         conda-forge       12MB
  + xorg-libice                      1.1.2  hb9d3cd8_0                    

In [9]:
# 3. Verificar las herramientas instaladas
!gmx --version
!acpype --version


                   :-) GROMACS - gmx, 2025.4-conda_forge (-:

Executable:   /usr/local/bin.AVX2_256/gmx
Data prefix:  /usr/local
Working dir:  /content
Command line:
  gmx --version

GROMACS version:     2025.4-conda_forge
Precision:           mixed
Memory model:        64 bit
MPI library:         thread_mpi
OpenMP support:      enabled (GMX_OPENMP_MAX_THREADS = 128)
GPU support:         OpenCL
NBNxM GPU setup:     super-cluster 2x2x2 / cluster 8 (cluster-pair splitting on)
SIMD instructions:   AVX2_256
CPU FFT library:     fftw-3.3.10-sse2-avx
GPU FFT library:     clFFT
Multi-GPU FFT:       none
RDTSCP usage:        disabled
TNG support:         enabled
Hwloc support:       disabled
Tracing support:     disabled
C compiler:          /home/conda/feedstock_root/build_artifacts/gromacs_1764344590343/_build_env/bin/x86_64-conda-linux-gnu-cc GNU 14.3.0
C compiler flags:    -fexcess-precision=fast -funroll-all-loops -mavx2 -mfma -Wno-missing-field-initializers -O3 -DNDEBUG
C++ compiler:    

## Paso 2: Generación de la Topología del Ligando con ACPYPE

Los campos de fuerza biológicos convencionales (como AMBER99SB-ILDN) no contienen parámetros predeterminados para moléculas orgánicas pequeñas como los ligandos. Utilizaremos el **Campo de Fuerzas General de Amber 2 (GAFF2)** acoplado al modelo de asignación de cargas semi-empírico **AM1-BCC**.

La herramienta **ACPYPE** (un wrapper de Python para `antechamber`) automatizará este proceso. Generaremos las topologías para **Afzelina (neutral)**, **Quercetina (neutral)** y **Ceftarolina (zwitteriónica/neutral)** a partir de sus archivos de estructura 3D en formato `.sdf` o `.pdb`.

> [!TIP]
> Si sus ligandos tienen cargas netas diferentes a cero, asegúrese de especificar el parámetro `-n` en `acpype` (por ejemplo, `-n -1` o `-n 1`). Para Quercetina y Afzelina, la carga neta es `0`.


In [10]:
# Crear directorio para el ligando y descargar/cargar los archivos SDF/PDB
!mkdir -p ligands
from google.colab import files
files.upload()

!mv *.pdb ligands/
!mv *.sdf ligands/

# NOTA: Puede cargar sus archivos SDF de ligandos directamente a la carpeta 'ligands/' en Colab.
# Para fines prácticos de automatización, aquí definiremos los comandos para ejecutar acpype.
# Asumiendo que tenemos los archivos de ligandos en el entorno:


Saving Afzelin_3D.sdf to Afzelin_3D.sdf
Saving Ceftaroline_3D.sdf to Ceftaroline_3D.sdf
Saving Quercetin_3D.sdf to Quercetin_3D.sdf
mv: cannot stat '*.pdb': No such file or directory


In [11]:
# Ejecutar ACPYPE para generar topología GAFF2 + AM1-BCC
# -i: Archivo de entrada
# -c: Método de cargas (bcc = AM1-BCC)
# -n: Carga neta del compuesto (0 por defecto)
# -m: Estilo de salida de GROMACS (amber)

print("=== GENERANDO TOPOLOGÍA PARA AFZELINA ===")
!acpype -i ligands/Afzelin_3D.sdf -c bcc -n 0 -m amber -f

print("=== GENERANDO TOPOLOGÍA PARA QUERCETINA ===")
!acpype -i ligands/Quercetin_3D.sdf -c bcc -n 0 -m amber -f

print("=== GENERANDO TOPOLOGÍA PARA CEFTAROLINA ===")
!acpype -i ligands/Ceftaroline_3D.sdf -c bcc -n 0 -m amber -f

print("Topologías generadas. Revise las carpetas creadas .acpype/")


=== GENERANDO TOPOLOGÍA PARA AFZELINA ===
usage: 
    acpype -i _file_ | _SMILES_string_ [-c _string_] [-n _int_] [-m _int_] [-a _string_] [-f] etc. or
    acpype -p _prmtop_ -x _inpcrd_ [-d | -w]

    output: assuming 'root' is the basename of either the top input file,
            the 3-letter residue name or user defined (-b option)
    root_bcc_gaff.mol2:  final mol2 file with 'bcc' charges and 'gaff' atom type
    root_AC.inpcrd    :  coord file for AMBER
    root_AC.prmtop    :  topology and parameter file for AMBER
    root_AC.lib       :  residue library file for AMBER
    root_AC.frcmod    :  modified force field parameters
    root_GMX.gro      :  coord file for GROMACS
    root_GMX.top      :  topology file for GROMACS
    root_GMX.itp      :  molecule unit topology and parameter file for GROMACS
    root_GMX_OPLS.itp :  OPLS/AA mol unit topol & par file for GROMACS (experimental!)
    em.mdp, md.mdp    :  run parameters file for GROMACS
    root_NEW.pdb      :  final pdb fi

## Paso 3: Generación de la Topología de la Proteína con GROMACS

Utilizaremos la herramienta nativa de GROMACS `gmx pdb2gmx` para procesar el archivo receptor PDB limpio, remover o regenerar los átomos de hidrógeno de manera consistente con el pH fisiológico (7.4), y generar la topología de la proteína utilizando el campo de fuerzas **AMBER99SB-ILDN** y el modelo de agua explícita **TIP3P**.


In [14]:
!mkdir receptors
print("Carga los archivos pdb de los receptores")
files.upload()

!mv *.pdb receptors/

mkdir: cannot create directory ‘receptors’: File exists
Carga los archivos pdb de los receptores


Saving PBP2a_3ZG0_clean.pdb to PBP2a_3ZG0_clean.pdb


In [15]:
# Comando estándar para generar la topología de la proteína
# -ff: Selecciona el campo de fuerzas AMBER99SB-ILDN
# -water: Selecciona el modelo de agua explícita TIP3P
# -ignh: Ignora los hidrógenos del PDB original para reconstruirlos de forma limpia y estándar

# Para MurG:
# !gmx pdb2gmx -f receptors/MurG_AF-Q6GGZ0-F1.pdb -o protein_processed.gro -ff amber99sb-ildn -water tip3p -ignh

# Para PBP2a:
!gmx pdb2gmx -f receptors/PBP2a_3ZG0_clean.pdb -o protein_processed.gro -ff amber99sb-ildn -water tip3p -ignh


               :-) GROMACS - gmx pdb2gmx, 2025.4-conda_forge (-:

Executable:   /usr/local/bin.AVX2_256/gmx
Data prefix:  /usr/local
Working dir:  /content
Command line:
  gmx pdb2gmx -f receptors/PBP2a_3ZG0_clean.pdb -o protein_processed.gro -ff amber99sb-ildn -water tip3p -ignh

Using the Amber99sb-ildn force field in directory amber99sb-ildn.ff

going to rename amber99sb-ildn.ff/aminoacids.r2b
Opening force field file /usr/local/share/gromacs/top/amber99sb-ildn.ff/aminoacids.r2b

going to rename amber99sb-ildn.ff/dna.r2b
Opening force field file /usr/local/share/gromacs/top/amber99sb-ildn.ff/dna.r2b

going to rename amber99sb-ildn.ff/rna.r2b
Opening force field file /usr/local/share/gromacs/top/amber99sb-ildn.ff/rna.r2b
Reading receptors/PBP2a_3ZG0_clean.pdb...
Read '', 10252 atoms

Analyzing pdb file
Splitting chemical chains based on TER records or chain id changing.

There are 2 chains and 0 blocks of water and 1277 residues with 10252 atoms

  chain  #res #atoms

  1 'A'   642  

## Paso 4: Ensamblaje del Complejo Proteína-Ligando

Para simular el complejo, debemos unificar los archivos de coordenadas `.gro` de la proteína procesada (`protein_processed.gro`) y del ligando generado por ACPYPE (`ligand_GMX.gro`). Posteriormente, debemos registrar el ligando en el archivo de topología principal de la proteína (`topol.top`).

### Script de Automatización de Fusión en Python
A continuación, definimos una función robusta de Python para fusionar las coordenadas `.gro` y otra para modificar `topol.top` insertando las definiciones del ligando en la ubicación exacta que requiere GROMACS (inmediatamente después de la definición de parámetros del campo de fuerzas, y antes de la definición de la proteína).


In [16]:
import os
def fusionar_gro(protein_gro, ligand_gro, output_gro):
    """
    Fusiona limpiamente las coordenadas de la proteína y del ligando en formato GRO.
    """
    if not os.path.exists(protein_gro) or not os.path.exists(ligand_gro):
        print("Error: Uno de los archivos GRO de entrada no existe.")
        return

    with open(protein_gro, 'r') as f:
        p_lines = f.readlines()
    with open(ligand_gro, 'r') as f:
        l_lines = f.readlines()

    p_atoms = int(p_lines[1].strip())
    l_atoms = int(l_lines[1].strip())
    total_atoms = p_atoms + l_atoms

    # Extraer los registros de átomos (excluyendo cabecera y caja de vectores)
    p_atom_lines = p_lines[2:-1]
    l_atom_lines = l_lines[2:-1]

    # Obtener el vector de caja de la proteína (última línea)
    box_vector = p_lines[-1]

    # Escribir el archivo unificado
    with open(output_gro, 'w') as f:
        f.write("Complejo Proteina-Ligando Generado Automáticamente\n")
        f.write(f"{total_atoms:>5}\n")
        for line in p_atom_lines:
            f.write(line)
        for line in l_atom_lines:
            f.write(line)
        f.write(box_vector)
    print(f"Fusión exitosa: {output_gro} con {total_atoms} átomos totales (Proteína: {p_atoms}, Ligando: {l_atoms})")

def actualizar_topol(topol_file, itp_file, ligand_resname):
    """
    Inserta la directiva #include de la topología del ligando en topol.top
    y lo añade en la sección final de moléculas.
    """
    if not os.path.exists(topol_file):
        print("Error: El archivo topol.top no existe.")
        return

    with open(topol_file, 'r') as f:
        lines = f.readlines()

    new_lines = []
    inserted_itp = False

    for line in lines:
        new_lines.append(line)
        # Insertar el include inmediatamente después del include del campo de fuerzas
        if 'forcefield.itp' in line and not inserted_itp:
            new_lines.append(f'#include "{itp_file}"\n')
            inserted_itp = True

    # Añadir el ligando a la sección final [ molecules ]
    new_lines.append(f"{ligand_resname:<20} 1\n")

    with open(topol_file, 'w') as f:
        f.writelines(new_lines)
    print(f"Topología topol.top actualizada con éxito para el ligando {ligand_resname}.")


In [ ]:
# Ejemplo de uso de la fusión en Colab:
# Asumiendo que el ligando se generó en 'Afzelin_3D.acpype/'
# fusionar_gro('protein_processed.gro', 'Afzelin_3D.acpype/Afzelin_3D_GMX.gro', 'complex.gro')
# actualizar_topol('topol.top', 'Afzelin_3D.acpype/Afzelin_3D_GMX.itp', 'LIG')


## Paso 5: Solvatación y Neutralización del Sistema

Definiremos el entorno físico de la simulación. En primer lugar, crearemos una caja de simulación de geometría **dodecaédrica** (la más eficiente computacionalmente, ahorrando ~30% de moléculas de agua en comparación con una cúbica) con una distancia mínima de **1.2 nm** de amortiguación de solvente desde los bordes de la proteína.

Posteriormente, solvaremos con agua explícita modelo **TIP3P** y agregaremos iones Na⁺ y Cl⁻ para neutralizar la carga eléctrica neta del sistema y alcanzar una **concentración fisiológica de 0.15 M** de NaCl.


In [ ]:
# 1. Definición de la caja del sistema
# -d 1.2: 1.2 nm de distancia a las paredes
# -bt dodecahedron: Caja dodecaédrica truncada para mayor eficiencia
# -c: Centra el complejo en la caja
!gmx editconf -f complex.gro -o complex_box.gro -c -d 1.2 -bt dodecahedron


In [ ]:
# 2. Solvatación con agua explícita TIP3P (utiliza el template spc216.gro)
!gmx solvate -cp complex_box.gro -cs spc216.gro -o complex_solv.gro -p topol.top


Para agregar iones con `gmx genion`, primero debemos crear un archivo de parámetros minimalista (`ions.mdp`) y compilar una estructura binaria portable `.tpr` temporal con `grompp`.


In [ ]:
# Escribir un archivo minimalista ions.mdp
with open('ions.mdp', 'w') as f:
    f.write("""\n
; Minimal mdp for genion\n
integrator      = steepest\n
emtol           = 1000.0\n
emstep          = 0.01\n
nsteps          = 50000\n
nstlist         = 1\n
cutoff-scheme   = Verlet\n
ns_type         = grid\n
coulombtype     = PME\n
rcoulomb        = 1.0\n
rvdw            = 1.0\n
pbc             = xyz\n
""")
print("ions.mdp creado.")


In [ ]:
# 3. Compilar el archivo tpr para iones
!gmx grompp -f ions.mdp -c complex_solv.gro -p topol.top -o ions.tpr -maxwarn 1

# 4. Reemplazar moléculas de solvente por iones Na+ y Cl- (0.15 M y neutro)
# Se canaliza 'SOL' para que reemplace únicamente moléculas de agua
!echo "SOL" | gmx genion -s ions.tpr -o complex_solv_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15


## Paso 6: Corrida de la Simulación MD (EM, NVT, NPT y Producción)

Ejecutaremos el protocolo de simulación estándar compuesto por cuatro etapas fundamentales:
1. **Minimización Energética (EM):** Método *Steepest Descent* para relajar choques estéricos espurios hasta alcanzar una fuerza máxima residual de 1000 kJ/mol/nm.
2. **Equilibración NVT (Isocórica-Isotérmica):** Calentamiento del sistema a **300 K** en 100 ps con posición restringida en átomos pesados utilizando un termostato de velocidad de reescalado modificada (*V-rescale*).
3. **Equilibración NPT (Isobárica-Isotérmica):** Ajuste de la densidad y presión del sistema a **1 bar** durante 100 ps utilizando el barostato de *Berendsen* (más robusto en equilibrio inicial) y posición restringida en átomos pesados.
4. **Producción MD:** Corrida de producción libre de **100 ns** sin restricciones, paso de integración de **2 fs**, utilizando el barostato de *Parrinello-Rahman* de alta fidelidad y el termostato *V-rescale*.

### Creación Automatizada de Parámetros `.mdp` en Google Colab


In [ ]:
# 1. Generar em.mdp
with open('em.mdp', 'w') as f:
    f.write("""\n
title                   = Energy Minimization\n
integrator              = steepest\n
emtol                   = 1000.0\n
emstep                  = 0.01\n
nsteps                  = 50000\n
nstlist                 = 1\n
cutoff-scheme           = Verlet\n
ns_type                 = grid\n
coulombtype             = PME\n
rcoulomb                = 1.0\n
rvdw                    = 1.0\n
pbc                     = xyz\n
dispcorr                = EnerPres\n
""")

# 2. Generar nvt.mdp
with open('nvt.mdp', 'w') as f:
    f.write("""\n
title                   = NVT Equilibration\n
define                  = -DPOSRES ; Position restraints on heavy atoms\n
integrator              = md\n
nsteps                  = 50000    ; 100 ps\n
dt                      = 0.002    ; 2 fs\n
nstxout                 = 500\n
nstvout                 = 500\n
nstenergy               = 500\n
nstlog                  = 500\n
cutoff-scheme           = Verlet\n
ns_type                 = grid\n
coulombtype             = PME\n
rcoulomb                = 1.0\n
rvdw                    = 1.0\n
tcoupl                  = V-rescale\n
tc-grps                 = System\n
tau_t                   = 0.1\n
ref_t                   = 300\n
pcoupl                  = no ; No pressure coupling\n
pbc                     = xyz\n
dispcorr                = EnerPres\n
constraints             = h-bonds\n
constraint_algorithm    = lincs\n
continuation            = no\n
""")

# 3. Generar npt.mdp
with open('npt.mdp', 'w') as f:
    f.write("""\n
title                   = NPT Equilibration\n
define                  = -DPOSRES ; Position restraints on heavy atoms\n
integrator              = md\n
nsteps                  = 50000    ; 100 ps\n
dt                      = 0.002    ; 2 fs\n
nstxout                 = 500\n
nstvout                 = 500\n
nstenergy               = 500\n
nstlog                  = 500\n
cutoff-scheme           = Verlet\n
ns_type                 = grid\n
coulombtype             = PME\n
rcoulomb                = 1.0\n
rvdw                    = 1.0\n
tcoupl                  = V-rescale\n
tc-grps                 = System\n
tau_t                   = 0.1\n
ref_t                   = 300\n
pcoupl                  = Berendsen\n
pcoupltype              = isotropic\n
tau_p                   = 2.0\n
ref_p                   = 1.0\n
compressibility         = 4.5e-5\n
refcoord_scaling        = com\n
pbc                     = xyz\n
dispcorr                = EnerPres\n
constraints             = h-bonds\n
constraint_algorithm    = lincs\n
continuation            = yes ; Continue from NVT\n
""")

# 4. Generar md.mdp (100 ns)
with open('md.mdp', 'w') as f:
    f.write("""\n
title                   = 100 ns Production MD\n
integrator              = md\n
nsteps                  = 50000000 ; 100 ns (50,000,000 steps of 2 fs)\n
dt                      = 0.002    ; 2 fs\n
nstxout                 = 0 ; Do not write coordinates to save space\n
nstvout                 = 0\n
nstenergy               = 5000 ; Every 10 ps\n
nstlog                  = 5000 ; Every 10 ps\n
nstxout-compressed      = 5000 ; Compressed coordinates (xtc) every 10 ps\n
compressed-x-grps       = System\n
cutoff-scheme           = Verlet\n
ns_type                 = grid\n
coulombtype             = PME\n
rcoulomb                = 1.0\n
rvdw                    = 1.0\n
tcoupl                  = V-rescale\n
tc-grps                 = System\n
tau_t                   = 0.1\n
ref_t                   = 300\n
pcoupl                  = Parrinello-Rahman\n
pcoupltype              = isotropic\n
tau_p                   = 2.0\n
ref_p                   = 1.0\n
compressibility         = 4.5e-5\n
pbc                     = xyz\n
dispcorr                = EnerPres\n
constraints             = h-bonds\n
constraint_algorithm    = lincs\n
continuation            = yes ; Continue from NPT\n
""")
print("Todos los archivos MDP de simulación han sido generados exitosamente.")


A continuación, compilamos y ejecutamos consecutivamente cada fase. Si utiliza una GPU de Colab, asegúrese de agregar el flag `-nb gpu` en el comando `mdrun`.


In [ ]:
print("=== FASE 1: MINIMIZACIÓN ENERGÉTICA ===")
!gmx grompp -f em.mdp -c complex_solv_ions.gro -p topol.top -o em.tpr
# Ejecutar la minimización. -v muestra el progreso en vivo
!gmx mdrun -v -deffnm em

print("=== FASE 2: EQUILIBRACIÓN NVT ===")
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr
!gmx mdrun -v -deffnm nvt

print("=== FASE 3: EQUILIBRACIÓN NPT ===")
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr
!gmx mdrun -v -deffnm npt

print("=== FASE 4: CORRIDA DE PRODUCCIÓN MD (100 ns) ===")
!gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md_100ns.tpr
# En Colab con GPU, mdrun aprovecha automáticamente los núcleos CUDA.
# Si tiene problemas con la GPU, remueva el flag '-nb gpu' o fuerce cpu con '-nb cpu'.
# !gmx mdrun -v -deffnm md_100ns -nb gpu


## Paso 7: Post-procesamiento y Análisis de la Trayectoria

Antes de realizar cualquier análisis métrico sobre la trayectoria, debemos eliminar los efectos artificiales de las **condiciones periódicas de contorno (PBC)**. Esto se hace centrando el soluto, rotando y trasladando la trayectoria para alinear el backbone de la proteína y evitar saltos moleculares espurios causados por la geometría periódica de la caja de simulación.


In [ ]:
# 1. Corregir saltos de condiciones de borde periódico (PBC)
# Se selecciona 'Protein' para centrado y 'System' para salida
!echo "Protein System" | gmx trjconv -s md_100ns.tpr -f md_100ns.xtc -o md_noPBC.xtc -pbc nojump -center

# 2. Realizar un ajuste por mínimos cuadrados rotacional y traslacional de la proteína
# Se selecciona 'Backbone' para el fit y 'System' o 'Protein_Ligand' para salida
!echo "Backbone System" | gmx trjconv -s md_100ns.tpr -f md_noPBC.xtc -o md_fit.xtc -fit rot+trans


### Cálculos Métricos Estructurales de Calidad
Calcularemos las cuatro métricas de referencia principales recomendadas por la literatura para evaluar la estabilidad:
1. **RMSD (Desviación Media Cuadrática):** Para verificar la convergencia del backbone del receptor (estabilidad global) y el movimiento del ligando.
2. **RMSF (Fluctuación Media Cuadrática):** Para caracterizar la flexibilidad residual de cada residuo de la proteína.
3. **Radio de Giro ($R_g$):** Mide el grado de compactación de la estructura terciaria tridimensional global de la proteína.
4. **Enlaces de Hidrógeno Proteína-Ligando ($H-bonds$):** Para evaluar la red de interacciones electrostáticas polares directas formadas en el bolsillo de unión.


In [ ]:
print("=== CALCULANDO RMSD ===")
# Backbone de la proteína:
!echo "Backbone Backbone" | gmx rms -s md_100ns.tpr -f md_fit.xtc -o rmsd_protein.xvg
# Ligando:
!echo "Backbone LIG" | gmx rms -s md_100ns.tpr -f md_fit.xtc -o rmsd_ligand.xvg

print("=== CALCULANDO RMSF POR RESIDUO ===")
!echo "C-alpha" | gmx rmsf -s md_100ns.tpr -f md_fit.xtc -o rmsf_residue.xvg -res

print("=== CALCULANDO RADIO DE GIRO (Rg) ===")
!echo "Protein" | gmx gyrate -s md_100ns.tpr -f md_fit.xtc -o gyrate_protein.xvg

print("=== CALCULANDO ENLACES DE HIDRÓGENO ===")
# Define el número de puentes de hidrógeno formados entre proteína y ligando:
!echo "Protein LIG" | gmx hbond -s md_100ns.tpr -f md_fit.xtc -num hbond_num.xvg


### Visualización de Resultados en Google Colab
Para acelerar la interpretación científica de las trayectorias sin necesidad de descargar los archivos de salida a un computador local, utilizaremos este script en Python que lee directamente los archivos de datos `.xvg` generados por GROMACS (filtrando los comentarios que inician con `@` y `&`) y los grafica utilizando **Matplotlib** y **Pandas** con formato listo para publicación científica.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def parse_xvg(filepath):
    """
    Función auxiliar para parsear archivos XVG de GROMACS limpiando cabeceras.
    """
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith(('@', '#', '&')) or not line.strip():
                continue
            parts = line.split()
            data.append([float(x) for x in parts])
    return np.array(data)

# 1. GRAFICAR RMSD
try:
    rmsd_prot = parse_xvg('rmsd_protein.xvg')
    rmsd_lig = parse_xvg('rmsd_ligand.xvg')

    plt.figure(figsize=(10, 5))
    # Convertir tiempo en ps a ns
    plt.plot(rmsd_prot[:, 0]/1000.0, rmsd_prot[:, 1]*10, label='Proteína (Backbone)', color='#1f77b4', linewidth=1.5)
    plt.plot(rmsd_lig[:, 0]/1000.0, rmsd_lig[:, 1]*10, label='Ligando (LIG)', color='#ff7f0e', linewidth=1.5)
    plt.title('Estabilidad del Complejo (Métrica RMSD)', fontsize=14, fontweight='bold')
    plt.xlabel('Tiempo (ns)', fontsize=12)
    plt.ylabel('RMSD (Å)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig('rmsd_plot.png', dpi=300)
    plt.show()
except Exception as e:
    print("Error al graficar RMSD:", e)

# 2. GRAFICAR RMSF POR RESIDUO
try:
    rmsf_data = parse_xvg('rmsf_residue.xvg')

    plt.figure(figsize=(12, 5))
    plt.plot(rmsf_data[:, 0], rmsf_data[:, 1]*10, color='#2ca02c', linewidth=1.5)
    plt.title('Fluctuación Residual por Aminoácido (Métrica RMSF)', fontsize=14, fontweight='bold')
    plt.xlabel('Número de Residuo', fontsize=12)
    plt.ylabel('RMSF (Å)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('rmsf_plot.png', dpi=300)
    plt.show()
except Exception as e:
    print("Error al graficar RMSF:", e)

# 3. GRAFICAR ENLACES DE HIDRÓGENO
try:
    hbond_data = parse_xvg('hbond_num.xvg')

    plt.figure(figsize=(10, 4))
    plt.plot(hbond_data[:, 0]/1000.0, hbond_data[:, 1], color='#d62728', linewidth=1.2)
    plt.title('Red de Enlaces de Hidrógeno Proteína-Ligando', fontsize=14, fontweight='bold')
    plt.xlabel('Tiempo (ns)', fontsize=12)
    plt.ylabel('Número de Enlaces H', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig('hbond_plot.png', dpi=300)
    plt.show()
except Exception as e:
    print("Error al graficar Enlaces de Hidrógeno:", e)
